# 01 – EDA: CIFAKE Dataset

**Dataset:** [birdy654/cifake-real-and-ai-generated-synthetic-images](https://www.kaggle.com/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images)

- REAL = CIFAR-10 photographs
- FAKE = Stable Diffusion v1.4
- Resolution: 32×32, balanced, 100k train / 20k test

In [ ]:
# --- Colab setup (run first) ---
# Runtime → Change runtime type → T4 GPU
!pip -q install kagglehub grad-cam

from pathlib import Path
from google.colab import drive
import kagglehub

# Everything we produce goes to Drive so it survives Colab restarts
drive.mount("/content/drive")
OUT = Path("/content/drive/MyDrive/ai-image-detector")
MODELS = OUT / "models"; REPORT = OUT / "report"
MODELS.mkdir(parents=True, exist_ok=True); REPORT.mkdir(parents=True, exist_ok=True)

# Downloads CIFAKE once (~100 MB); contains train/{REAL,FAKE} and test/{REAL,FAKE}
DATA_ROOT = Path(kagglehub.dataset_download("birdy654/cifake-real-and-ai-generated-synthetic-images"))
print("Data:", DATA_ROOT, "| Outputs:", OUT)

In [ ]:
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from collections import Counter

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


assert DATA_ROOT.exists(), f"Dataset not found at {DATA_ROOT}"

In [ ]:
# Expected structure:
# train/REAL, train/FAKE, test/REAL, test/FAKE
splits = {}
for split in ["train", "test"]:
    for cls in ["REAL", "FAKE"]:
        folder = DATA_ROOT / split / cls
        files = list(folder.glob("*.jpg")) + list(folder.glob("*.png"))
        splits[f"{split}/{cls}"] = files
        print(f"{split}/{cls}: {len(files):,} images")

In [ ]:
# Class balance bar chart
labels = list(splits.keys())
counts = [len(v) for v in splits.values()]
colors = ["#22C55E" if "REAL" in l else "#8B5CF6" for l in labels]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(labels, counts, color=colors, edgecolor="none")
ax.set_ylabel("Count")
ax.set_title("CIFAKE class counts")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(REPORT / "eda_class_counts.png", dpi=150)
plt.show()

In [ ]:
# Sample grid: 4 REAL + 4 FAKE from train
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i, cls in enumerate(["REAL", "FAKE"]):
    samples = random.sample(splits[f"train/{cls}"], 4)
    for j, path in enumerate(samples):
        img = Image.open(path)
        axes[i, j].imshow(img)
        axes[i, j].set_title(cls, fontsize=9, color="#22C55E" if cls == "REAL" else "#8B5CF6")
        axes[i, j].axis("off")
plt.suptitle("Sample grid (32×32 native)", fontsize=12)
plt.tight_layout()
plt.savefig(REPORT / "eda_sample_grid.png", dpi=150)
plt.show()

In [ ]:
# Pixel intensity histograms (sanity check for generation artifacts)
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, cls, color in zip(axes, ["REAL", "FAKE"], ["#22C55E", "#8B5CF6"]):
    samples = random.sample(splits[f"train/{cls}"], 200)
    pixels = []
    for p in samples:
        arr = np.array(Image.open(p).convert("L")).ravel()
        pixels.append(arr)
    hist, bins = np.histogram(np.concatenate(pixels), bins=64, range=(0, 255), density=True)
    ax.bar(bins[:-1], hist, width=4, color=color, alpha=0.8)
    ax.set_title(f"{cls} intensity distribution")
    ax.set_xlabel("Pixel value")
plt.tight_layout()
plt.savefig(REPORT / "eda_intensity.png", dpi=150)
plt.show()

## Notes for the report

- Dataset is balanced → no class-weighting needed.
- 32×32 is tiny; EfficientNet will upscale to 224. This blurs fine artifacts and limits Grad-CAM spatial detail.
- FAKE images come from a single generator (SD v1.4) → expect accuracy drop on Midjourney / Flux / DALL·E.